In [9]:
import os
os.chdir(r"C:\Users\14731\desktop\Capstone\Wifi-indoor-localization-with-rtt-and-csi\DL")
print("当前工作目录:", os.getcwd())

当前工作目录: C:\Users\14731\desktop\Capstone\Wifi-indoor-localization-with-rtt-and-csi\DL


In [31]:
import pandas as pd
import numpy as np
import ast

def align_csi_ftm_from_csv(ftm_csv, csi_csv, window_ms=100):
    """
    对齐 CSI 和 FTM 数据（支持 datetime 时间戳，保留微秒）。
    返回：
        - csi_list: 每条 CSI 的 I/Q 数据加权平均
        - rssi_list: 对应的 RSSI
        - rtt_list: 对应的 RTT (ns)
    window_ms: 对齐时间窗口，单位毫秒
    """
    # 读取 CSV
    ftm_df = pd.read_csv(ftm_csv)
    csi_df = pd.read_csv(csi_csv)

    # 统一列名
    ftm_df.rename(columns=lambda x: x.strip().lower(), inplace=True)
    csi_df.rename(columns=lambda x: x.strip().lower(), inplace=True)

    # 转换 timestamp 为 datetime
    ftm_df['timestamp'] = pd.to_datetime(ftm_df['timestamp'], errors='coerce')
    csi_df['timestamp'] = pd.to_datetime(csi_df['timestamp'], errors='coerce')

    # 删除无法解析的行
    ftm_df.dropna(subset=['timestamp'], inplace=True)
    csi_df.dropna(subset=['timestamp'], inplace=True)

    # 初始化输出列表
    csi_list = []
    rssi_list = []
    rtt_list = []

    window = pd.Timedelta(milliseconds=window_ms)

    for _, ftm_row in ftm_df.iterrows():
        ftm_time = ftm_row['timestamp']

        # 找到时间窗口内的 CSI
        mask = (csi_df['timestamp'] >= ftm_time - window) & \
               (csi_df['timestamp'] <= ftm_time + window)
        csi_window = csi_df[mask]

        if len(csi_window) == 0:
            continue

        # 时间差加权
        times = csi_window['timestamp'].values.astype('datetime64[ns]').astype(np.float64)  # ns -> float
        ftm_ns = np.datetime64(ftm_time).astype('datetime64[ns]').astype(np.float64)
        dts = np.abs(times - ftm_ns)
        weights = 1 / (dts + 1e-9)  # 避免除零
        weights /= weights.sum()

        # 解析 CSI data
        csi_data_all = []
        for row in csi_window['data']:
            if isinstance(row, str):
                csi_data_all.append(ast.literal_eval(row))
            else:
                csi_data_all.append(row)
        csi_data_all = np.array(csi_data_all, dtype=np.float32)

        # 加权平均
        csi_avg = np.average(csi_data_all, axis=0, weights=weights)

        # 保存
        csi_list.append(csi_avg.tolist())
        rssi_list.append(float(csi_window['rssi'].iloc[0]))  # 取窗口第一条的 RSSI
        rtt_list.append(float(ftm_row['rtt_raw (nsec)']))

    return csi_list, rssi_list, rtt_list


# ==============================
# 用法示例
# ==============================
csi_data, rssi_data, rtt_data = align_csi_ftm_from_csv(
    ftm_csv="data/ftm_data.csv",
    csi_csv="data/csi_data.csv",
    window_ms=100
)

print("对齐后的样本数:", len(csi_data))
print("第一个 CSI:", csi_data[0])
print("第一个 RSSI:", rssi_data[0])
print("第一个 RTT:", rtt_data[0])

for csi in csi_data:
    print(len(csi))

对齐后的样本数: 74
第一个 CSI: [0.0, 0.0, -3.0521494009866106, 9.221987315010571, -2.7336152219873155, 8.540521494009868, -2.7336152219873155, 8.540521494009868, -2.7336152219873155, 8.540521494009868, -2.7336152219873155, 8.177589852008458, -2.4150810429880205, 8.177589852008458, -2.4150810429880205, 7.496124031007752, -2.0965468639887246, 7.814658210007047, -1.4150810429880198, 7.133192389006342, -1.0965468639887248, 7.451726568005637, -0.7336152219873152, 7.451726568005637, -0.4150810429880201, 7.770260747004933, -0.4150810429880201, 7.770260747004933, -0.4150810429880201, 7.088794926004229, 0.26638477801268495, 7.770260747004933, -0.052149400986610146, 7.407329105003524, -0.052149400986610146, 7.770260747004933, -0.052149400986610146, 7.770260747004933, -0.3706835799859061, 8.451726568005638, -1.0521494009866112, 8.133192389006343, -0.7336152219873152, 8.133192389006343, -1.4150810429880198, 7.814658210007047, -2.0965468639887246, 7.496124031007752, -2.7780126849894295, 7.496124031007752, -2

In [41]:
import numpy as np

def preprocess_csi(iq_list):
    """
    将原始 CSI I/Q 数据转换为 [幅度, 相位] 格式，并归一化
    参数：
        iq_list: list 或 np.ndarray，形如 [I0, Q0, I1, Q1, ...]
    返回：
        features: np.ndarray, shape = (num_subcarriers * 2,)
                  格式为 [amp0, phase0, amp1, phase1, ...]
    """
    iq_array = np.array(iq_list, dtype=np.float32)
    assert len(iq_array) % 2 == 0, "CSI 数据长度必须是偶数（I/Q 成对）"
    
    # 拆分 I / Q
    I = iq_array[0::2]
    Q = iq_array[1::2]
    
    # 计算幅度和相位
    amplitude = np.sqrt(I**2 + Q**2)
    phase = np.arctan2(Q, I)
    
    # 相位解缠（unwrap）
    phase = np.unwrap(phase)
    
    # 幅度归一化（0~1）
    if amplitude.max() > 0:
        amplitude = amplitude / amplitude.max()
    
    # 相位归一化到 [-1, 1]（原始范围大约是 -π ~ π）
    phase = phase / np.pi
    
    # 拼接为 [amp0, phase0, amp1, phase1, ...]
    features = np.empty(amplitude.size * 2, dtype=np.float32)
    features[0::2] = amplitude
    features[1::2] = phase
    
    return features

features = preprocess_csi(csi_data[0])

print("原始 CSI 长度:", len(csi_data[0]))
print("预处理后特征长度:", len(features))
print("特征:", features)


原始 CSI 长度: 128
预处理后特征长度: 128
特征: [0.         0.         0.97879153 0.6017372  0.9035626  0.5986034
 0.9035626  0.5986034  0.9035626  0.5986034  0.86880517 0.6026879
 0.85916895 0.59140784 0.79355365 0.5992098  0.81526214 0.5834327
 0.73275787 0.5623369  0.7589332  0.5465066  0.75447714 0.5312366
 0.7840596  0.5169877  0.7840596  0.5169877  0.7155011  0.5186172
 0.7834032  0.48909178 0.7463922  0.50224096 0.7829609  0.50213623
 0.7829609  0.50213623 0.85242754 0.5139518  0.8263416  0.54095066
 0.8228398  0.5286341  0.8002224  0.55702174 0.78430647 0.5868081
 0.8055203  0.6129687  0.6999022  0.61520565 0.6983009  0.649726
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.7022264  0.5671571
 0.6999022  0.61520565 0.7342167  0.6095917  0.77405983 0.6486936
 0.7455427  0.654865   0.78001285 0.6796599  0.81114644 0.67198

In [34]:
class DistanceDataset:
    def __init__(self, csi_list, rssi_list, rtt_list, distance_list,
                 rssi_range=(-100, 0), rtt_max=300.0, dist_max=30.0):
        """
        csi_list: list of list, 每个样本的原始 I/Q 数据
        rssi_list: list of float, 单位 dBm
        rtt_list: list of float, 单位 ns
        distance_list: list of float, 单位 m
        """
        self.rssi_min, self.rssi_max = rssi_range
        self.rtt_max = rtt_max
        self.dist_max = dist_max

        # 预处理并保存
        self.csi_data = [preprocess_csi(csi) for csi in csi_list]
        self.rssi_data = [self.normalize_rssi(r) for r in rssi_list]
        self.rtt_data = [self.normalize_rtt(t) for t in rtt_list]
        self.distance_data = [self.normalize_distance(d) for d in distance_list]

    def normalize_rssi(self, rssi):
        return (rssi - self.rssi_min) / (self.rssi_max - self.rssi_min)

    def normalize_rtt(self, rtt):
        return rtt / self.rtt_max

    def normalize_distance(self, dist):
        return dist / self.dist_max

    def denormalize_distance(self, norm_dist):
        return norm_dist * self.dist_max

    def get_batch(self, indices):
        csi_batch = torch.tensor([self.csi_data[i] for i in indices], dtype=torch.float32)
        rssi_batch = torch.tensor([[self.rssi_data[i]] for i in indices], dtype=torch.float32)
        rtt_batch = torch.tensor([[self.rtt_data[i]] for i in indices], dtype=torch.float32)
        dist_batch = torch.tensor([[self.distance_data[i]] for i in indices], dtype=torch.float32)
        return csi_batch, rssi_batch, rtt_batch, dist_batch

In [32]:
import torch
import torch.nn as nn

class MLPDistanceModel(nn.Module):
    def __init__(self, csi_dim):
        super().__init__()
        # CSI 分支
        self.csi_branch = nn.Sequential(
            nn.Linear(csi_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        # RTT+RSSI 分支
        self.other_branch = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU()
        )
        # 融合
        self.fc = nn.Sequential(
            nn.Linear(32 + 16, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # 输出距离
        )

    def forward(self, csi, rtt, rssi):
        csi_feat = self.csi_branch(csi)
        other_feat = self.other_branch(torch.cat([rtt, rssi], dim=1))
        x = torch.cat([csi_feat, other_feat], dim=1)
        return self.fc(x)

model = MLPDistanceModel(csi_dim=128)
print(model)

MLPDistanceModel(
  (csi_branch): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (other_branch): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
  )
  (fc): Sequential(
    (0): Linear(in_features=48, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [37]:
N = 100
CSI_DIM = 128
csi_list = csi_data
rssi_list = rssi_data
rtt_list = rtt_data
dist_list = np.random.uniform(0.3,0.3,size=N).tolist()

dataset = DistanceDataset(csi_list, rssi_list, rtt_list, dist_list)

csi_batch, rssi_batch, rtt_batch, dist_batch = dataset.get_batch([0,1,2])

print("csi_batch shape:", csi_batch.shape)
print("csi_batch[0]:", csi_batch[0])
print("rssi_batch:", rssi_batch)
print("rtt_batch:", rtt_batch)
print("dist_batch:", dist_batch)

csi_batch shape: torch.Size([3, 128])
csi_batch[0]: tensor([0.0000, 0.0000, 0.9788, 0.6017, 0.9036, 0.5986, 0.9036, 0.5986, 0.9036,
        0.5986, 0.8688, 0.6027, 0.8592, 0.5914, 0.7936, 0.5992, 0.8153, 0.5834,
        0.7328, 0.5623, 0.7589, 0.5465, 0.7545, 0.5312, 0.7841, 0.5170, 0.7841,
        0.5170, 0.7155, 0.5186, 0.7834, 0.4891, 0.7464, 0.5022, 0.7830, 0.5021,
        0.7830, 0.5021, 0.8524, 0.5140, 0.8263, 0.5410, 0.8228, 0.5286, 0.8002,
        0.5570, 0.7843, 0.5868, 0.8055, 0.6130, 0.6999, 0.6152, 0.6983, 0.6497,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.7022, 0.5672, 0.6999, 0.6152, 0.7342,
        0.6096, 0.7741, 0.6487, 0.7455, 0.6549, 0.7800, 0.6797, 0.8111, 0.6720,
        0.8111, 0.6720, 0.8224, 0.7012, 0.7482, 0.7074, 0.8039, 0.6909, 0.8305,
        0.6840, 0.7740, 0.6994, 0.7531, 0.6869, 0.7531, 0.6869, 0.76

In [ ]:
model = MLPDistanceModel(csi_dim=len(dataset.csi_data[0]))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
batch_size = 16
epochs = 20

In [ ]:
for epoch in range(epochs):
    perm = np.random.permutation(len(dataset))
    total_loss = 0
    for i in range(0, len(dataset), batch_size):
        indices = perm[i:i+batch_size]
        csi_batch, rssi_batch, rtt_batch, dist_batch = dataset.get_batch(indices)

        pred = model(csi_batch, rtt_batch, rssi_batch)  
        loss = criterion(pred, dist_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(indices)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataset):.6f}")